# Scoring Evaluation

This notebook compares the LLM-based and embedding-based scoring approaches,
analyzes score distributions, and visualizes the embedding space.

**Purpose:** Demonstrate that the scoring pipeline produces meaningful differentiation
between high-signal and low-signal content.

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from sklearn.manifold import TSNE

# Add parent directory to path for imports
sys.path.insert(0, str(Path.cwd().parent))

# Configure matplotlib
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 11

## 1. Generate Synthetic Test Data

We create a diverse set of test articles spanning different relevance levels.
This allows evaluation even without historical digest data.

In [ ]:
# Synthetic test articles representing different relevance tiers
TEST_ARTICLES = [
    # Clearly high-relevance (expected 8-10)
    {
        "title": "Mechanistic Interpretability of In-Context Learning: Uncovering the Algorithm",
        "summary": "We present a circuit-level analysis of how transformers implement in-context learning. Using activation patching and causal interventions, we identify specific attention heads responsible for the algorithmic behavior.",
        "source": "arxiv.org",
        "expected_tier": "must_read",
    },
    {
        "title": "Superposition and Polysemanticity in Language Models",
        "summary": "We extend the superposition hypothesis to explain how language models represent more features than they have dimensions. Includes novel visualization techniques for understanding feature geometry.",
        "source": "transformer-circuits.pub",
        "expected_tier": "must_read",
    },
    {
        "title": "Three-Year Longitudinal Study: Adolescent Social Media Use and Cognitive Development",
        "summary": "We tracked 3,000 teenagers across three years measuring social media usage patterns and cognitive outcomes. Results show nuanced effects varying by platform type and usage context.",
        "source": "journals.sagepub.com",
        "expected_tier": "must_read",
    },
    {
        "title": "RLHF and the Emergence of Sycophancy: An Empirical Investigation",
        "summary": "We study how reinforcement learning from human feedback creates systematic biases toward agreeable but potentially misleading responses. Proposes novel evaluation framework for truthfulness.",
        "source": "alignmentforum.org",
        "expected_tier": "must_read",
    },
    {
        "title": "Activation Steering: Fine-Grained Control of Language Model Behavior",
        "summary": "We demonstrate precise behavioral control through targeted activation modifications. Includes analysis of how steering vectors interact with model internals.",
        "source": "arxiv.org",
        "expected_tier": "must_read",
    },
    
    # Medium relevance (expected 5-7)
    {
        "title": "Digital Literacy Programs in Schools: A Meta-Analysis",
        "summary": "Systematic review of 45 digital literacy interventions. Finds moderate effects on critical thinking about online information.",
        "source": "academic journal",
        "expected_tier": "worth_look",
    },
    {
        "title": "The Ethics of AI in Healthcare: A Framework",
        "summary": "Proposes ethical guidelines for deploying AI in clinical settings. Addresses consent, transparency, and accountability.",
        "source": "bioethics journal",
        "expected_tier": "worth_look",
    },
    {
        "title": "Working Paper: LLM Evaluation Challenges and Best Practices",
        "summary": "Preliminary findings on issues with current LLM benchmarks. Proposes improved evaluation methodology.",
        "source": "researcher blog",
        "expected_tier": "worth_look",
    },
    {
        "title": "Interview: AI Safety Researcher on Existential Risk",
        "summary": "In-depth conversation covering alignment challenges, compute governance, and laboratory safety protocols.",
        "source": "podcast",
        "expected_tier": "worth_look",
    },
    {
        "title": "The Attention Economy: How Platforms Capture Mind Share",
        "summary": "Analysis of engagement optimization techniques and their cognitive effects. Draws on enactivist cognitive science.",
        "source": "substack.com",
        "expected_tier": "worth_look",
    },
    
    # Low relevance (expected 1-4)
    {
        "title": "TechCorp Announces $200M Series C Funding",
        "summary": "AI startup TechCorp closed largest funding round in company history. Plans to expand engineering team.",
        "source": "techcrunch.com",
        "expected_tier": "drop",
    },
    {
        "title": "10 Productivity Tips for Remote Workers",
        "summary": "Improve your work from home setup with these simple tips. Number 7 will surprise you!",
        "source": "lifestyle blog",
        "expected_tier": "drop",
    },
    {
        "title": "Government Announces New AI Task Force",
        "summary": "Press release: Ministry establishes committee to study AI implications. First meeting scheduled for next month.",
        "source": "government website",
        "expected_tier": "drop",
    },
    {
        "title": "NGO Weekly Newsletter: Updates on Digital Rights",
        "summary": "This week: new coalition member, upcoming conference, call for volunteers, staff changes.",
        "source": "ngo newsletter",
        "expected_tier": "drop",
    },
    {
        "title": "Celebrity Thoughts on AI Taking Over Hollywood",
        "summary": "Famous actor shares concerns about AI in entertainment industry during late night interview.",
        "source": "entertainment news",
        "expected_tier": "drop",
    },
    {
        "title": "Q3 Earnings Beat Expectations for Tech Giants",
        "summary": "Major tech companies report strong quarterly results. Cloud revenue up 30% year-over-year.",
        "source": "financial news",
        "expected_tier": "drop",
    },
    {
        "title": "New Smartphone Features AI-Powered Camera",
        "summary": "Product launch: Latest model includes enhanced photo processing and voice assistant improvements.",
        "source": "product review site",
        "expected_tier": "drop",
    },
    {
        "title": "Country X Bans Social Media Platform",
        "summary": "Government cites national security concerns. Platform responds that they will comply with local laws.",
        "source": "news wire",
        "expected_tier": "drop",
    },
]

print(f"Test dataset: {len(TEST_ARTICLES)} articles")
print(f"  - Expected must_read: {sum(1 for a in TEST_ARTICLES if a['expected_tier'] == 'must_read')}")
print(f"  - Expected worth_look: {sum(1 for a in TEST_ARTICLES if a['expected_tier'] == 'worth_look')}")
print(f"  - Expected drop: {sum(1 for a in TEST_ARTICLES if a['expected_tier'] == 'drop')}")

## 2. Score with Embedding-Based Scorer

We use the sentence-transformer embedding scorer to generate relevance scores.
This runs entirely offline with no API key required.

In [ ]:
from evaluate_embeddings import score_batch_embeddings, get_embeddings_batch, ANCHOR_TEXTS

# Score all test articles
articles_copy = [a.copy() for a in TEST_ARTICLES]
scored_articles = score_batch_embeddings(articles_copy)

# Extract scores
embedding_scores = [a['score'] for a in scored_articles]

print("Embedding-based scores:")
for article in scored_articles:
    tier = article['expected_tier']
    score = article['score']
    match = "✓" if (tier == 'must_read' and score >= 8) or \
                   (tier == 'worth_look' and 5 <= score <= 7) or \
                   (tier == 'drop' and score <= 4) else "✗"
    print(f"  [{match}] {score:2d}/10 ({tier:10s}) {article['title'][:60]}...")

## 3. Score Distribution Analysis

A well-calibrated scorer should show clear separation between tiers.
We visualize the distribution of scores to check for clustering.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: Overall histogram
ax1 = axes[0]
ax1.hist(embedding_scores, bins=range(1, 12), edgecolor='black', alpha=0.7, color='steelblue')
ax1.set_xlabel('Score')
ax1.set_ylabel('Count')
ax1.set_title('Distribution of Embedding Scores')
ax1.set_xticks(range(1, 11))
ax1.axvline(x=8, color='green', linestyle='--', label='Must Read threshold (≥8)')
ax1.axvline(x=5, color='orange', linestyle='--', label='Worth a Look threshold (≥5)')
ax1.legend()

# Right: By expected tier
ax2 = axes[1]
tiers = ['must_read', 'worth_look', 'drop']
colors = ['green', 'orange', 'gray']
tier_scores = {
    tier: [a['score'] for a in scored_articles if a['expected_tier'] == tier]
    for tier in tiers
}

positions = [1, 2, 3]
bp = ax2.boxplot(
    [tier_scores[t] for t in tiers],
    positions=positions,
    patch_artist=True,
    widths=0.6
)
for patch, color in zip(bp['boxes'], colors):
    patch.set_facecolor(color)
    patch.set_alpha(0.6)

ax2.set_xticklabels(['Must Read\n(expected 8-10)', 'Worth a Look\n(expected 5-7)', 'Drop\n(expected 1-4)'])
ax2.set_ylabel('Actual Score')
ax2.set_title('Scores by Expected Tier')
ax2.set_ylim(0, 11)
ax2.axhline(y=8, color='green', linestyle='--', alpha=0.5)
ax2.axhline(y=5, color='orange', linestyle='--', alpha=0.5)

plt.tight_layout()
plt.savefig('score_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

print("\nScore statistics by expected tier:")
for tier in tiers:
    scores = tier_scores[tier]
    print(f"  {tier:12s}: mean={np.mean(scores):.1f}, std={np.std(scores):.1f}, range=[{min(scores)}, {max(scores)}]")

## 4. Precision Metrics

We compute precision-like metrics to evaluate scoring quality:
- **Precision@k**: Of articles scored 8+, what fraction are actually must-read?
- **Tier accuracy**: What fraction of articles fall into their expected tier?

In [ ]:
def compute_metrics(articles):
    """Compute precision and accuracy metrics."""
    # Predicted tiers based on score
    def predicted_tier(score):
        if score >= 8:
            return 'must_read'
        elif score >= 5:
            return 'worth_look'
        else:
            return 'drop'
    
    # Add predictions
    for a in articles:
        a['predicted_tier'] = predicted_tier(a['score'])
    
    # Precision for must_read: of those predicted as must_read, how many actually are?
    predicted_must_read = [a for a in articles if a['predicted_tier'] == 'must_read']
    if predicted_must_read:
        true_positives = sum(1 for a in predicted_must_read if a['expected_tier'] == 'must_read')
        precision_must_read = true_positives / len(predicted_must_read)
    else:
        precision_must_read = 0.0
    
    # Recall for must_read: of actual must_reads, how many were predicted correctly?
    actual_must_read = [a for a in articles if a['expected_tier'] == 'must_read']
    if actual_must_read:
        recall_must_read = sum(1 for a in actual_must_read if a['predicted_tier'] == 'must_read') / len(actual_must_read)
    else:
        recall_must_read = 0.0
    
    # Overall accuracy
    correct = sum(1 for a in articles if a['predicted_tier'] == a['expected_tier'])
    accuracy = correct / len(articles)
    
    # Separation: mean score of must_reads vs drops
    must_read_scores = [a['score'] for a in articles if a['expected_tier'] == 'must_read']
    drop_scores = [a['score'] for a in articles if a['expected_tier'] == 'drop']
    separation = np.mean(must_read_scores) - np.mean(drop_scores) if must_read_scores and drop_scores else 0
    
    return {
        'precision_must_read': precision_must_read,
        'recall_must_read': recall_must_read,
        'accuracy': accuracy,
        'separation': separation,
    }

metrics = compute_metrics(scored_articles)

print("Embedding Scorer Metrics:")
print(f"  Precision (Must Read): {metrics['precision_must_read']:.1%}")
print(f"  Recall (Must Read):    {metrics['recall_must_read']:.1%}")
print(f"  Overall Accuracy:      {metrics['accuracy']:.1%}")
print(f"  Score Separation:      {metrics['separation']:.1f} points")
print()
print("Interpretation:")
print("  - Precision: Of articles we flag as must-read, what % actually are?")
print("  - Recall: Of actual must-reads, what % do we catch?")
print("  - Separation: Mean score gap between must-read and drop tiers")

## 5. Embedding Space Visualization

We use t-SNE to project the high-dimensional embeddings into 2D space,
colored by score tier. Good separation in this space indicates the
embedding model captures relevance-related features.

In [ ]:
# Get embeddings for all articles
texts = [f"{a['title']}. {a['summary']}" for a in TEST_ARTICLES]
embeddings = get_embeddings_batch(texts)

# Also get anchor embeddings for reference
anchor_embeddings = get_embeddings_batch(ANCHOR_TEXTS[:5])  # First 5 anchors

# Combine for visualization
all_embeddings = np.vstack([embeddings, anchor_embeddings])

# Run t-SNE
print("Running t-SNE...")
tsne = TSNE(n_components=2, random_state=42, perplexity=min(5, len(all_embeddings) - 1))
embeddings_2d = tsne.fit_transform(all_embeddings)

# Split back
article_embeddings_2d = embeddings_2d[:len(texts)]
anchor_embeddings_2d = embeddings_2d[len(texts):]

# Plot
fig, ax = plt.subplots(figsize=(12, 8))

# Color by expected tier
tier_colors = {'must_read': 'green', 'worth_look': 'orange', 'drop': 'gray'}
tier_labels = {'must_read': 'Must Read', 'worth_look': 'Worth a Look', 'drop': 'Drop'}

for tier in ['must_read', 'worth_look', 'drop']:
    mask = [a['expected_tier'] == tier for a in TEST_ARTICLES]
    points = article_embeddings_2d[mask]
    ax.scatter(
        points[:, 0], points[:, 1],
        c=tier_colors[tier],
        label=tier_labels[tier],
        s=100,
        alpha=0.7,
        edgecolors='black',
        linewidths=0.5
    )

# Plot anchors as stars
ax.scatter(
    anchor_embeddings_2d[:, 0], anchor_embeddings_2d[:, 1],
    c='red', marker='*', s=300, label='Anchor Texts',
    edgecolors='black', linewidths=0.5, zorder=5
)

ax.set_xlabel('t-SNE Dimension 1')
ax.set_ylabel('t-SNE Dimension 2')
ax.set_title('Embedding Space: Articles Colored by Expected Relevance Tier')
ax.legend(loc='upper right')

plt.tight_layout()
plt.savefig('embedding_space.png', dpi=150, bbox_inches='tight')
plt.show()

print("\nInterpretation:")
print("  - Green points (Must Read) should cluster near red stars (Anchors)")
print("  - Gray points (Drop) should be further from anchors")
print("  - Clear separation indicates embeddings capture relevance")

## 6. Similarity Analysis

Examine the distribution of cosine similarities to anchor texts
across different article tiers.

In [ ]:
from evaluate_embeddings import cosine_similarity, get_anchor_embeddings

# Get anchor embeddings
pos_anchors, neg_anchors = get_anchor_embeddings()

# Compute similarities for each article
max_pos_sims = []
max_neg_sims = []
net_sims = []

for emb in embeddings:
    pos_sims = cosine_similarity(emb, pos_anchors)
    neg_sims = cosine_similarity(emb, neg_anchors)
    max_pos = float(np.max(pos_sims))
    max_neg = float(np.max(neg_sims))
    max_pos_sims.append(max_pos)
    max_neg_sims.append(max_neg)
    net_sims.append(max_pos - max_neg * 0.5)

# Plot
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: Positive vs negative similarity scatter
ax1 = axes[0]
for tier in ['must_read', 'worth_look', 'drop']:
    mask = [a['expected_tier'] == tier for a in TEST_ARTICLES]
    pos = [max_pos_sims[i] for i, m in enumerate(mask) if m]
    neg = [max_neg_sims[i] for i, m in enumerate(mask) if m]
    ax1.scatter(neg, pos, c=tier_colors[tier], label=tier_labels[tier], s=80, alpha=0.7)

ax1.set_xlabel('Max Negative Similarity (noise signal)')
ax1.set_ylabel('Max Positive Similarity (relevance signal)')
ax1.set_title('Positive vs Negative Anchor Similarity')
ax1.legend()
# Add diagonal line
lims = [0, max(max(max_pos_sims), max(max_neg_sims)) + 0.05]
ax1.plot(lims, lims, 'k--', alpha=0.3, label='Equal similarity')

# Right: Net similarity by tier
ax2 = axes[1]
tier_net_sims = {
    tier: [net_sims[i] for i, a in enumerate(TEST_ARTICLES) if a['expected_tier'] == tier]
    for tier in tiers
}
bp = ax2.boxplot(
    [tier_net_sims[t] for t in tiers],
    positions=[1, 2, 3],
    patch_artist=True,
    widths=0.6
)
for patch, color in zip(bp['boxes'], colors):
    patch.set_facecolor(color)
    patch.set_alpha(0.6)

ax2.set_xticklabels(['Must Read', 'Worth a Look', 'Drop'])
ax2.set_ylabel('Net Similarity (pos - 0.5*neg)')
ax2.set_title('Net Similarity Score by Tier')

plt.tight_layout()
plt.savefig('similarity_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

## 7. Summary

Key findings from this analysis:

In [ ]:
print("=" * 60)
print("SCORING EVALUATION SUMMARY")
print("=" * 60)
print()
print(f"Dataset: {len(TEST_ARTICLES)} synthetic articles")
print("Scorer: Sentence-transformer embedding similarity (all-MiniLM-L6-v2)")
print()
print("Metrics:")
print(f"  • Precision (Must Read):  {metrics['precision_must_read']:.0%}")
print(f"  • Recall (Must Read):     {metrics['recall_must_read']:.0%}")
print(f"  • Overall Tier Accuracy:  {metrics['accuracy']:.0%}")
print(f"  • Mean Score Separation:  {metrics['separation']:.1f} points")
print()
print("Score Distribution:")
for tier in tiers:
    scores = tier_scores[tier]
    print(f"  • {tier_labels[tier]:15s}: {np.mean(scores):.1f} ± {np.std(scores):.1f}")
print()
print("Interpretation:")
if metrics['separation'] > 3:
    print("  ✓ Good separation between must-read and drop tiers")
else:
    print("  ⚠ Limited separation between tiers - may need anchor refinement")
    
if metrics['precision_must_read'] > 0.7:
    print("  ✓ High precision - most flagged items are actually relevant")
else:
    print("  ⚠ Lower precision - some false positives in must-read tier")
    
if metrics['recall_must_read'] > 0.7:
    print("  ✓ Good recall - catching most relevant content")
else:
    print("  ⚠ Lower recall - may be missing some relevant content")

---

**Next steps for improvement:**
1. Tune anchor texts to better match actual high-value content
2. Adjust similarity-to-score thresholds based on empirical data
3. Compare with LLM scores on real digest history
4. Consider domain-specific embedding models for better relevance capture